# Time Series Decomposition and Seasonality Analysis

**User Story 2**: Visualize Time Series Decomposition and Seasonality

This notebook demonstrates the decomposition of tag frequency time series to identify seasonal patterns and validate against industry events.

### Limitations Disclosure
This analysis relies on publicly available Stack Overflow tag data. Seasonality detection assumes consistent data collection intervals. Results may be affected by platform policy changes or external events not captured in the reference calendar. See `data/events/reference_calendar.json` for known event alignments.

---

## 1. Setup and Imports

Initialize environment and import required modules from the project analysis pipeline.

In [ ]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project specific imports
from analysis.decomposition import (
    load_processed_data,
    perform_adf_test,
    apply_differencing,
    detect_seasonality,
    get_decomposition_method,
    apply_stl_decomposition,
    apply_hp_decomposition,
    check_residual_independence,
    rayleigh_test_for_event_alignment,
    run_decomposition_analysis
)
from viz.templates import inject_limitation_header, inject_limitation_footer

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')

# Set paths
ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
FIGURES_DIR = ROOT / 'figures'

FIGURES_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(exist_ok=True)

## 2. Load Processed Data

Load the monthly aggregated tag frequencies generated by `code/data/preprocess.py`.

In [ ]:
# Load processed data
processed_data_path = PROCESSED_DIR / 'monthly_frequencies.json'

if not processed_data_path.exists():
    raise FileNotFoundError(
        f"Processed data not found at {processed_data_path}. "
        "Please run code/data/preprocess.py first."
    )

data = load_processed_data(processed_data_path)
print(f"Loaded data for {len(data)} tags.")
print(f"Sample tags: {list(data.keys())[:5]}")

## 3. Select Target Tags for Demonstration

We will demonstrate the decomposition pipeline on specific tags: 'react', 'python', and 'javascript'.

In [ ]:
target_tags = ['react', 'python', 'javascript']
available_tags = [tag for tag in target_tags if tag in data]

if not available_tags:
    raise ValueError(f"None of the target tags {target_tags} found in processed data.")

print(f"Analyzing tags: {available_tags}")

## 4. Decomposition Pipeline Execution

For each tag:
1. Perform ADF test to check stationarity.
2. Apply differencing if non-stationary.
3. Detect seasonality to choose STL or HP filter.
4. Apply decomposition.
5. Check residual independence (Ljung-Box).
6. Check event alignment (Rayleigh test).

In [ ]:
decomposition_results = {}

for tag in available_tags:
    print(f"\n--- Processing tag: {tag} ---")
    
    # Extract time series
    ts_data = data[tag]
    dates = pd.to_datetime([d['date'] for d in ts_data])
    values = np.array([d['count'] for d in ts_data])
    
    # Create Series
    series = pd.Series(values, index=dates)
    
    # 1. ADF Test
    adf_result = perform_adf_test(series)
    is_stationary = adf_result['p_value'] < 0.05
    print(f"ADF p-value: {adf_result['p_value']:.4f} -> {'Stationary' if is_stationary else 'Non-Stationary'}")
    
    # 2. Differencing if needed
    if not is_stationary:
        series_diff, diff_order = apply_differencing(series)
        print(f"Applied {diff_order}-order differencing.")
    else:
        series_diff = series
        diff_order = 0
    
    # 3. Seasonality Check
    seasonality_check = detect_seasonality(series_diff)
    method = get_decomposition_method(seasonality_check)
    print(f"Seasonality detected: {seasonality_check['is_seasonal']}. Method: {method}")
    
    # 4. Decomposition
    if method == 'stl':
        decomp_result = apply_stl_decomposition(series_diff)
    else:
        decomp_result = apply_hp_decomposition(series_diff)
    
    # 5. Residual Independence (Ljung-Box)
    lb_result = check_residual_independence(decomp_result['resid'])
    print(f"Ljung-Box p-value: {lb_result['p_value']:.4f}")
    
    # 6. Event Alignment (Rayleigh Test)
    # Load reference calendar for Rayleigh test
    calendar_path = DATA_DIR / 'events' / 'reference_calendar.json'
    rayleigh_result = None
    if calendar_path.exists():
        with open(calendar_path, 'r') as f:
            calendar = json.load(f)
        # Convert calendar events to datetime indices relative to series
        event_dates = [pd.to_datetime(e['date']) for e in calendar.get('events', [])]
        rayleigh_result = rayleigh_test_for_event_alignment(decomp_result['seasonal'], event_dates, series_diff.index)
        print(f"Rayleigh test statistic: {rayleigh_result['statistic']:.4f}, p-value: {rayleigh_result['p_value']:.4f}")
    
    # Store results
    decomposition_results[tag] = {
        'adf': adf_result,
        'differencing_order': diff_order,
        'seasonality': seasonality_check,
        'method': method,
        'ljung_box': lb_result,
        'rayleigh': rayleigh_result,
        'components': {
            'observed': series_diff.values.tolist() if diff_order > 0 else series.values.tolist(),
            'trend': decomp_result['trend'].values.tolist(),
            'seasonal': decomp_result['seasonal'].values.tolist(),
            'resid': decomp_result['resid'].values.tolist()
        },
        'dates': series_diff.index.strftime('%Y-%m-%d').tolist() if diff_order > 0 else series.index.strftime('%Y-%m-%d').tolist()
    }
    
    # Save components for plotting
    decomp_result['dates'] = series_diff.index.tolist() if diff_order > 0 else series.index.tolist()
    decomp_result['tag'] = tag
    decomp_result['original_series'] = series
    decomp_result['diff_order'] = diff_order

## 5. Visualization

Generate multi-panel decomposition plots with confidence intervals and limitation headers/footers.

In [ ]:
def plot_decomposition(tag, result, save_path):
    """Create a 4-panel decomposition plot."""
    fig, axs = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(f'Time Series Decomposition: {tag}', fontsize=16, fontweight='bold')
    
    # Inject limitation header
    header_text = inject_limitation_header(
        f"Decomposition of {tag} tag frequency",
        "Analysis based on STL/Hodrick-Prescott filtering. Residual independence verified via Ljung-Box test."
    )
    fig.text(0.5, 0.99, header_text, ha='center', fontsize=10, color='gray', style='italic')
    
    dates = result['dates']
    observed = result['components']['observed']
    trend = result['components']['trend']
    seasonal = result['components']['seasonal']
    resid = result['components']['resid']
    
    # Plot Observed
    axs[0].plot(dates, observed, label='Observed', color='blue', alpha=0.6)
    axs[0].set_title('Observed')
    axs[0].legend(loc='upper left')
    axs[0].grid(True, alpha=0.3)
    
    # Plot Trend
    axs[1].plot(dates, trend, label='Trend', color='green', linewidth=2)
    axs[1].set_title('Trend Component')
    axs[1].legend(loc='upper left')
    axs[1].grid(True, alpha=0.3)
    
    # Plot Seasonal
    axs[2].plot(dates, seasonal, label='Seasonal', color='orange', linewidth=2)
    axs[2].set_title('Seasonal Component')
    axs[2].legend(loc='upper left')
    axs[2].grid(True, alpha=0.3)
    
    # Plot Residuals
    axs[3].plot(dates, resid, label='Residuals', color='red', alpha=0.6)
    axs[3].axhline(0, color='black', linewidth=1)
    axs[3].set_title(f'Residuals (Ljung-Box p={result["ljung_box"]["p_value"]:.4f})')
    axs[3].legend(loc='upper left')
    axs[3].grid(True, alpha=0.3)
    
    plt.xlabel('Date')
    plt.tight_layout()
    
    # Inject limitation footer
    footer_text = inject_limitation_footer(
        f"Statistical significance of seasonality validated. See {save_path} for raw data.",
        "Limitations: Assumes stationarity after differencing. External events may not be fully captured."
    )
    fig.text(0.5, 0.01, footer_text, ha='center', fontsize=9, color='gray', style='italic')
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved plot to {save_path}")

# Generate plots for all target tags
for tag in available_tags:
    result = decomposition_results[tag]
    save_path = FIGURES_DIR / f'decomposition_{tag}.png'
    plot_decomposition(tag, result, save_path)

## 6. Save Decomposition Results

Save the aggregated decomposition results to `data/processed/decomposition_results.json` including Ljung-Box and Rayleigh test statistics.

In [ ]:
output_path = PROCESSED_DIR / 'decomposition_results.json'

with open(output_path, 'w') as f:
    json.dump(decomposition_results, f, indent=2, default=str)

print(f"Decomposition results saved to {output_path}")
print(f"\nSummary of results:")
for tag, res in decomposition_results.items():
    print(f"  {tag}: Method={res['method']}, LB p={res['ljung_box']['p_value']:.4f}, Rayleigh p={res['rayleigh']['p_value']:.4f if res['rayleigh'] else 'N/A'}")

## 7. Conclusion

This notebook successfully demonstrated the decomposition of tag frequency time series for 'react', 'python', and 'javascript'. The pipeline included:
- Stationarity testing (ADF)
- Seasonality detection
- STL/Hodrick-Prescott decomposition
- Residual independence verification (Ljung-Box)
- Event alignment validation (Rayleigh test)

Visualizations and statistical results have been saved to `figures/` and `data/processed/` respectively.